# Agentic AI Systems — Worklet 15
# Executive Decision Support System
## Lab 5 · MCP-Style Connector — one server for all sources

**Phase A · Foundations** &nbsp;|&nbsp; **Goal:** expose every data source through one typed connector.

---

### What Lab 5 contributes to Worklet 15

| | |
|---|---|
| **Inputs** | The worklet's data sources. |
| **Outputs** | A uniform, typed tool surface. |
| **Agents / nodes** | Connector used by all data-facing agents. |
| **Deliverable** | A connector server with typed endpoints and auth; agents call only through it. |
| **Acceptance** | **All data access goes through the connector; swapping a source needs no agent change.** |
| **Pitfall to avoid** | No agent should touch a source directly; a bypass defeats governance later. |

> Chapter 15 pseudocode:
> ```
> connector.expose:
>     read('institutional KPIs & dashboards')
>     read('department reports')
>     read('external benchmarks')
> agents.use(connector)   # no agent touches a source directly
> ```

### Three real backends, not three wrappers

The lab reference is *"one server exposing File + SQLite + API"*. So:

| Resource | Backend | What it really is |
|---|---|---|
| `department_reports` | `file:csv` | a CSV **written to disk** and read with `csv.DictReader` |
| `kpi_definitions` | `sqlite` | a real **`sqlite3` database**, queried with SQL |
| `external_benchmark` | `http:benchmark-api` | a mock HTTP service with **latency** and HTTP-style failures |

One envelope, one error vocabulary, one audit log across all three.

### The two things this lab is really graded on

1. **Cell 6 — the swap.** `department_reports` is re-pointed from the CSV file to the SQLite table. The same agent function is re-run **untouched** and every figure is identical. Provenance changes — correctly — because the rows genuinely came from somewhere else.
2. **Cell 7 — the bypass.** An agent reads the CSV directly and gets a numerically correct answer with **no scope check, no provenance and no log entry**. Because every legitimate read *is* logged, that bypass is detectable — which is what makes the governance in Lab 12 enforceable rather than aspirational.

### How to run

Top to bottom, **no API key and no network**. Cell 3 writes real files into `lab5_sources/`. Cell 8 executes the rubric as assertions.

---

### Cell 1 · The shared foundation

The Labs 1–2 engine, unchanged, written to `edss_core.py` so this notebook runs standalone.

*(Collapse this cell — prior work, restated only for self-containment.)*

In [1]:
%%writefile edss_core.py
"""edss_core — shared foundation for the Worklet 15 lab notebooks.

Worklet 15 · Executive Decision Support System · Agentic AI (26ECAC401)

This is the Lab 1 + Lab 2 foundation, unchanged: the institutional data, the
typed tool contracts, the single ToolRouter choke point, the Connector, and the
validation gate. Labs 3-8 build on top of it rather than restating it, so every
notebook works against the same tools and the same numbers.

Written to disk by the first code cell of each lab notebook, so each notebook
runs standalone in Colab with no setup.
"""

from __future__ import annotations

import csv
import io
import json
import math
import re
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

# 1 · Institutional data (identical to Lab 2, Cell 3)
# =============================================================================

REPORTING_PERIOD = "academic year 2025-26"

DEPARTMENT_REPORTS_CSV = """dept_code,dept_name,category,students_appeared,students_passed,co_po_attainment,sanctioned_faculty,publications,citations,grants_lakh,eligible_students,placed_students,total_package_lpa,interns_completed,budget_allocated_lakh,budget_spent_lakh,figure_status
CSE,Computer Science & Engineering,circuit,240,231,2.71,34,58,402,96.5,228,212,2247.2,205,420,393,final
ECE,Electronics & Communication,circuit,180,168,2.58,26,34,221,41.0,172,143,1058.2,149,310,276,final
ISE,Information Science & Engineering,circuit,120,116,2.66,18,21,140,22.5,114,103,947.6,96,240,221,final
EEE,Electrical & Electronics,circuit,120,109,2.41,17,16,88,18.0,116,86,524.6,84,210,181,final
MECH,Mechanical Engineering,core,180,158,2.33,24,19,95,27.5,174,111,599.4,102,300,247,final
CIVIL,Civil Engineering,core,120,101,2.19,16,11,47,12.0,118,62,285.2,61,190,149,final
CHEM,Chemical Engineering,core,60,54,2.28,11,14,76,15.5,58,33,168.3,34,140,118,final
BT,Biotechnology,core,60,55,2.35,10,12,63,9.5,57,29,142.1,30,130,102,final
"""

NUMERIC_COLUMNS = {
    "students_appeared": int, "students_passed": int, "co_po_attainment": float,
    "sanctioned_faculty": int, "publications": int, "citations": int,
    "grants_lakh": float, "eligible_students": int, "placed_students": int,
    "total_package_lpa": float, "interns_completed": int,
    "budget_allocated_lakh": float, "budget_spent_lakh": float,
}

KPI_DEFINITIONS: Dict[str, Dict[str, Any]] = {
    "KPI-A1": {"name": "Institution pass percentage", "domain": "Academic",
               "method": "ratio", "numerator": "students_passed",
               "denominator": "students_appeared", "scale": 100, "unit": "%",
               "formula": "sum(students_passed) / sum(students_appeared) x 100",
               "target": 92.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Academic ERP"},
    "KPI-A2": {"name": "CO/PO attainment index", "domain": "Academic",
               "method": "weighted_mean", "value_field": "co_po_attainment",
               "weight_field": "students_appeared", "unit": "index 0-3",
               "formula": "sum(co_po_attainment x students_appeared) / sum(students_appeared)",
               "target": 2.50, "target_source": "NAAC criterion 2",
               "direction": "higher_is_better", "source_system": "Academic ERP (OBE module)"},
    "KPI-R1": {"name": "Publications per faculty", "domain": "Research",
               "method": "ratio", "numerator": "publications",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(publications) / sum(sanctioned_faculty)",
               "target": 1.50, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R2": {"name": "Citations per faculty", "domain": "Research",
               "method": "ratio", "numerator": "citations",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(citations) / sum(sanctioned_faculty)",
               "target": 9.00, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R3": {"name": "Grants sanctioned", "domain": "Research",
               "method": "sum", "value_field": "grants_lakh", "unit": "INR lakh",
               "formula": "sum(grants_lakh)",
               "target": 300.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-P1": {"name": "Placement percentage", "domain": "Placement",
               "method": "ratio", "numerator": "placed_students",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(placed_students) / sum(eligible_students) x 100",
               "target": 80.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P2": {"name": "Average package", "domain": "Placement",
               "method": "ratio", "numerator": "total_package_lpa",
               "denominator": "placed_students", "scale": 1, "unit": "LPA",
               "formula": "sum(total_package_lpa) / sum(placed_students)",
               "target": 8.00, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P3": {"name": "Internship completion", "domain": "Placement",
               "method": "ratio", "numerator": "interns_completed",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(interns_completed) / sum(eligible_students) x 100",
               "target": 85.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-F1": {"name": "Budget utilisation", "domain": "Finance",
               "method": "ratio", "numerator": "budget_spent_lakh",
               "denominator": "budget_allocated_lakh", "scale": 100, "unit": "%",
               "formula": "sum(budget_spent_lakh) / sum(budget_allocated_lakh) x 100",
               "target": 90.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Finance & Accounts ledger",
               "figure_status": "provisional",
               "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes "
                         "books in August."},
}

BENCHMARK_API_DB = {
    "KPI-R1": {"peer_median": 1.82, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-R2": {"peer_median": 11.40, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-P1": {"peer_median": 78.60, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
}

DOMAINS = ["Academic", "Research", "Placement", "Finance"]
DOMAIN_KPIS = {d: [k for k, v in KPI_DEFINITIONS.items() if v["domain"] == d] for d in DOMAINS}

# Registrar-authorised read access, per Gate 1 of the hybrid execution methodology.
ACCESS_REGISTER = {
    "Academic":  {"source_system": "Academic ERP — examination & OBE records",
                  "read_access": "granted by Registrar on 2026-06-01", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["pass percentage", "CO/PO attainment", "curriculum compliance"]},
    "Research":  {"source_system": "Research & Publications repository (Scopus-synced)",
                  "read_access": "granted by Registrar on 2026-06-01", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["publications", "citations", "grants sanctioned"]},
    "Placement": {"source_system": "Placement Cell tracker",
                  "read_access": "granted by Registrar on 2026-06-04", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["placement percentage", "average package",
                              "internship completion"],
                  "caveat": "3 provisional offers are excluded until the Placement Cell "
                            "verifies them."},
    "Finance":   {"source_system": "Finance & Accounts ledger",
                  "read_access": "granted by Registrar on 2026-06-07",
                  "figure_status": "provisional",
                  # 2025-26 is deliberately absent: the books are not closed, and the
                  # Lab 1 planner must notice that rather than plan over it.
                  "periods": ["2023-24", "2024-25"],
                  "metrics": ["budget utilisation", "revenue", "grant funds"],
                  "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes "
                            "books in August."},
}


# =============================================================================
# 2 · Tool contracts (identical to Lab 2, Cell 4)
# =============================================================================

TOOL_SCHEMAS: Dict[str, Dict[str, Any]] = {
    "read_department_reports": {
        "description": "Read per-department metric rows for one reporting period.",
        "parameters": {"type": "object", "properties": {
            "period": {"type": "string"},
            "departments": {"type": "array", "items": {"type": "string"}},
            "category": {"type": "string", "enum": ["circuit", "core"]}},
            "required": ["period"]},
        "returns": "{rows, row_count, source_system} | {error}"},
    "read_kpi_definitions": {
        "description": "Read approved KPI definitions: formula, unit, target, authority.",
        "parameters": {"type": "object", "properties": {
            "kpi_ids": {"type": "array", "items": {"type": "string"}}}, "required": []},
        "returns": "{definitions, count} | {error}"},
    "read_external_benchmark": {
        "description": "Published peer median for a KPI. Coverage is partial.",
        "parameters": {"type": "object", "properties": {"kpi_id": {"type": "string"}},
                       "required": ["kpi_id"]},
        "returns": "{kpi_id, peer_median, cohort, year} | {error}"},
    "filter_rows": {
        "description": "Filter department rows by an exact field value; errors on empty.",
        "parameters": {"type": "object", "properties": {
            "rows": {"type": "array", "items": {"type": "object"}},
            "field": {"type": "string"}, "value": {"type": "string"}},
            "required": ["rows", "field", "value"]},
        "returns": "{rows, row_count, filter} | {error}"},
    "compute_kpi": {
        "description": "Compute one approved KPI over department rows, with evidence.",
        "parameters": {"type": "object", "properties": {
            "kpi_id": {"type": "string"},
            "rows": {"type": "array", "items": {"type": "object"}}},
            "required": ["kpi_id", "rows"]},
        "returns": "{kpi_id, value, unit, formula, evidence, source_system} | {error}"},
    "compare_to_target": {
        "description": "Compare a KPI value to its approved target; flags rounding risk.",
        "parameters": {"type": "object", "properties": {
            "kpi_id": {"type": "string"}, "value": {"type": "number"}},
            "required": ["kpi_id", "value"]},
        "returns": "{value, target, variance, status, rounding_warning} | {error}"},
    "compute_variance": {
        "description": "Difference between a KPI value and any reference figure.",
        "parameters": {"type": "object", "properties": {
            "value": {"type": "number"}, "reference": {"type": "number"},
            "label": {"type": "string"}}, "required": ["value", "reference", "label"]},
        "returns": "{variance, abs_variance, pct_of_reference} | {error}"},
    "draft_notification": {
        "description": "DRAFT a leadership notification. Never sends. Sources required.",
        "parameters": {"type": "object", "properties": {
            "audience": {"type": "string"}, "subject": {"type": "string"},
            "findings": {"type": "array", "items": {"type": "object"}}},
            "required": ["audience", "subject", "findings"]},
        "returns": "{status: 'DRAFT - NOT SENT', body, requires_approval_from} | {error}"},
}


def _typed_rows(text: str) -> List[Dict[str, Any]]:
    rows = []
    for raw in csv.DictReader(io.StringIO(text)):
        row = dict(raw)
        for col, caster in NUMERIC_COLUMNS.items():
            if col in row:
                try:
                    row[col] = caster(row[col])
                except (TypeError, ValueError):
                    return [{"error": f"column '{col}' is not {caster.__name__}"}]
        rows.append(row)
    return rows


# Optional demo latency, so the parallel/sequential distinction in the run
# timeline is visible. Off by default; never affects any computed figure.
SOURCE_LATENCY_SECONDS = 0.0


def read_department_reports(period: str, departments=None, category=None) -> Dict[str, Any]:
    if period != REPORTING_PERIOD:
        return {"error": f"no department report export for '{period}'; the only loaded "
                         f"period is '{REPORTING_PERIOD}'"}
    if SOURCE_LATENCY_SECONDS:
        time.sleep(SOURCE_LATENCY_SECONDS)
    rows = _typed_rows(DEPARTMENT_REPORTS_CSV)
    if rows and "error" in rows[0]:
        return {"error": rows[0]["error"]}
    if departments:
        known = {r["dept_code"] for r in rows}
        unknown = [d for d in departments if d not in known]
        if unknown:
            return {"error": f"unknown department code(s): {unknown}; known: {sorted(known)}"}
        rows = [r for r in rows if r["dept_code"] in departments]
    if category:
        rows = [r for r in rows if r["category"] == category]
        if not rows:
            return {"error": f"no departments in category '{category}'"}
    if not rows:
        return {"error": "empty department report export"}
    return {"rows": rows, "row_count": len(rows), "period": period,
            "source_system": "Department reports export (Registrar-authorised read)"}


def read_kpi_definitions(kpi_ids=None) -> Dict[str, Any]:
    if kpi_ids:
        unknown = [k for k in kpi_ids if k not in KPI_DEFINITIONS]
        if unknown:
            return {"error": f"unknown KPI id(s): {unknown}"}
        defs = {k: KPI_DEFINITIONS[k] for k in kpi_ids}
    else:
        defs = dict(KPI_DEFINITIONS)
    return {"definitions": defs, "count": len(defs),
            "source_system": "Institutional KPI register (approved definitions)"}


def read_external_benchmark(kpi_id: str) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    record = BENCHMARK_API_DB.get(kpi_id)
    if record is None:
        return {"error": f"no published external benchmark for {kpi_id} "
                         f"({KPI_DEFINITIONS[kpi_id]['name']}) in the benchmark API"}
    return {"kpi_id": kpi_id, **record, "source_system": "External benchmark API (NIRF cohort)"}


def filter_rows(rows, field: str, value: str) -> Dict[str, Any]:
    if not rows:
        return {"error": "filter_rows received an empty row set"}
    if field not in rows[0]:
        return {"error": f"unknown field '{field}'; available: {sorted(rows[0])}"}
    subset = [r for r in rows if str(r[field]) == str(value)]
    if not subset:
        return {"error": f"filter {field}={value!r} matched 0 of {len(rows)} rows — "
                         f"refusing to return an empty subset"}
    return {"rows": subset, "row_count": len(subset), "filter": f"{field}=={value}",
            "filtered_from": len(rows)}


def compute_kpi(kpi_id: str, rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Generic: the arithmetic comes from the KPI definition, not from this code."""
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    if not rows:
        return {"error": f"cannot compute {kpi_id} over an empty row set"}
    d = KPI_DEFINITIONS[kpi_id]
    method, evidence = d["method"], []
    try:
        if method == "ratio":
            num_f, den_f, scale = d["numerator"], d["denominator"], d["scale"]
            num = sum(r[num_f] for r in rows)
            den = sum(r[den_f] for r in rows)
            if den == 0:
                return {"error": f"{kpi_id}: denominator sum({den_f}) is zero"}
            value = num / den * scale
            inputs = {f"sum({num_f})": round(num, 4), f"sum({den_f})": round(den, 4)}
            for r in rows:
                dv = (r[num_f] / r[den_f] * scale) if r[den_f] else None
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[num_f], "denominator": r[den_f],
                                 "numerator_field": num_f, "denominator_field": den_f,
                                 "dept_value": round(dv, 2) if dv is not None else None,
                                 "share_of_denominator_pct": round(r[den_f] / den * 100, 2)})
        elif method == "weighted_mean":
            vf, wf = d["value_field"], d["weight_field"]
            wsum = sum(r[wf] for r in rows)
            if wsum == 0:
                return {"error": f"{kpi_id}: total weight sum({wf}) is zero"}
            value = sum(r[vf] * r[wf] for r in rows) / wsum
            inputs = {f"sum({vf} x {wf})": round(sum(r[vf] * r[wf] for r in rows), 4),
                      f"sum({wf})": wsum}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[vf], "denominator": r[wf],
                                 "numerator_field": vf, "denominator_field": wf,
                                 "dept_value": r[vf],
                                 "share_of_denominator_pct": round(r[wf] / wsum * 100, 2)})
        elif method == "sum":
            vf = d["value_field"]
            value = sum(r[vf] for r in rows)
            inputs = {f"sum({vf})": round(value, 4)}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[vf], "denominator": None,
                                 "numerator_field": vf, "denominator_field": None,
                                 "dept_value": r[vf],
                                 "share_of_denominator_pct": round(r[vf] / value * 100, 2)
                                 if value else None})
        else:
            return {"error": f"unsupported KPI method '{method}' for {kpi_id}"}
    except KeyError as missing:
        return {"error": f"{kpi_id}: department rows are missing column {missing}"}

    statuses = {r.get("figure_status", "final") for r in rows}
    figure_status = d.get("figure_status") or ("final" if statuses == {"final"} else "mixed")
    out = {"kpi_id": kpi_id, "name": d["name"], "domain": d["domain"],
           "value": round(value, 4), "unit": d["unit"], "formula": d["formula"],
           "method": method, "n_departments": len(rows),
           "departments": [r["dept_code"] for r in rows],
           "inputs_used": inputs, "evidence": evidence,
           "source_system": d["source_system"], "figure_status": figure_status}
    if d.get("caveat"):
        out["caveat"] = d["caveat"]
    return out


def compare_to_target(kpi_id: str, value: float) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    d = KPI_DEFINITIONS[kpi_id]
    target = d["target"]
    variance = value - target
    pct_of_target = value / target * 100 if target else None
    gap_pct = abs(variance) / target * 100 if target else 0
    status = "on target" if variance >= 0 else ("at risk" if gap_pct <= 2.0 else "below target")
    rounding_warning = None
    if variance < 0:
        for places in (0, 1):
            if round(value, places) >= target:
                rounding_warning = (f"{value:.4g} rounds to {round(value, places)} at "
                                    f"{places} decimal place(s), which would read as meeting "
                                    f"the target of {target}. Report the unrounded figure.")
                break
    return {"kpi_id": kpi_id, "name": d["name"], "value": round(value, 4), "target": target,
            "target_source": d["target_source"], "unit": d["unit"],
            "variance": round(variance, 4),
            "pct_of_target": round(pct_of_target, 2) if pct_of_target else None,
            "status": status, "rounding_warning": rounding_warning}


def compute_variance(value: float, reference: float, label: str) -> Dict[str, Any]:
    if reference == 0:
        return {"error": f"cannot compute variance against a zero reference ({label})"}
    variance = value - reference
    return {"value": round(value, 4), "reference": round(reference, 4), "label": label,
            "variance": round(variance, 4), "abs_variance": round(abs(variance), 4),
            "pct_of_reference": round(value / reference * 100, 2)}


def draft_notification(audience: str, subject: str, findings: List[Dict[str, Any]]):
    if not findings:
        return {"error": "refusing to draft a notification with no findings"}
    unsourced = [f for f in findings if not f.get("source")]
    if unsourced:
        return {"error": f"{len(unsourced)} finding(s) carry no source; every figure in a "
                         f"leadership notification must cite where it came from"}
    body = "\n".join(f"  {i}. {f['statement']}\n     source: {f['source']}"
                     for i, f in enumerate(findings, 1))
    return {"status": "DRAFT - NOT SENT", "audience": audience, "subject": subject,
            "body": body, "findings_cited": len(findings),
            "requires_approval_from": "Principal / Registrar (Chapter 15 human gate)",
            "note": "This tool cannot send. Circulation happens only after human approval."}


TOOLS = {
    "read_department_reports": read_department_reports,
    "read_kpi_definitions": read_kpi_definitions,
    "read_external_benchmark": read_external_benchmark,
    "filter_rows": filter_rows,
    "compute_kpi": compute_kpi,
    "compare_to_target": compare_to_target,
    "compute_variance": compute_variance,
    "draft_notification": draft_notification,
}


# =============================================================================
# 3 · Router, connector, validation gate (Lab 2, Cells 5-6)
# =============================================================================

class ToolContractError(Exception):
    """Raised before a call when arguments do not satisfy the tool schema."""


class ToolError(Exception):
    """Raised after a call when the tool returned an explicit error."""


_JSON_TYPES = {"string": str, "number": (int, float), "integer": int,
               "boolean": bool, "array": list, "object": dict}


def _summarise(obj, _depth=0):
    if isinstance(obj, list):
        if len(obj) > 3 and all(isinstance(x, dict) for x in obj):
            return f"[{len(obj)} rows: {', '.join(str(x.get('dept_code', '?')) for x in obj)}]"
        return [_summarise(x, _depth + 1) for x in obj]
    if isinstance(obj, dict):
        return {k: _summarise(v, _depth + 1) for k, v in obj.items()}
    if isinstance(obj, str) and len(obj) > 160:
        return obj[:157] + "..."
    return obj


class ToolRouter:
    """The single choke point. Validates, executes, logs, raises on error."""

    def __init__(self, agent: str = "Coordinator"):
        self.agent = agent
        self.trace: List[Dict[str, Any]] = []
        self.raw_outputs: List[Any] = []

    def validate_input(self, tool: str, kwargs: Dict[str, Any]) -> None:
        if tool not in TOOL_SCHEMAS:
            raise ToolContractError(f"no such tool: '{tool}'")
        schema = TOOL_SCHEMAS[tool]["parameters"]
        props, required = schema["properties"], schema.get("required", [])
        for name in required:
            if name not in kwargs:
                raise ToolContractError(f"{tool}: missing required argument '{name}'")
        for name, val in kwargs.items():
            if name not in props:
                raise ToolContractError(
                    f"{tool}: unknown argument '{name}'; accepts {sorted(props)}")
            spec = props[name]
            expected = _JSON_TYPES.get(spec["type"])
            if expected and not isinstance(val, expected):
                raise ToolContractError(
                    f"{tool}.{name}: expected {spec['type']}, got {type(val).__name__}")
            if spec.get("enum") and val not in spec["enum"]:
                raise ToolContractError(f"{tool}.{name}: {val!r} not in {spec['enum']}")

    def call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        self.validate_input(tool, kwargs)
        started = time.perf_counter()
        result = TOOLS[tool](**kwargs)
        entry = {"step": len(self.trace) + 1, "agent": self.agent, "tool": tool,
                 "reason": reason, "input": _summarise(kwargs), "output": _summarise(result),
                 "ok": "error" not in result,
                 "ms": round((time.perf_counter() - started) * 1000, 2),
                 "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        self.trace.append(entry)
        self.raw_outputs.append(result)
        if "error" in result:
            raise ToolError(f"{tool} failed: {result['error']}")
        return result

    def try_call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        try:
            return self.call(tool, reason, **kwargs)
        except ToolError:
            return dict(TOOLS[tool](**kwargs))


class Connector:
    """One governed surface over every source. No agent touches a source directly."""

    RESOURCES = {
        "department reports":              "read_department_reports",
        "institutional KPIs & dashboards": "read_kpi_definitions",
        "external benchmarks":             "read_external_benchmark",
    }

    def __init__(self, router: ToolRouter):
        self.router = router

    def read(self, resource: str, reason: str, required: bool = True, **params):
        if resource not in self.RESOURCES:
            raise ToolContractError(f"connector exposes no resource '{resource}'; "
                                    f"available: {sorted(self.RESOURCES)}")
        caller = self.router.call if required else self.router.try_call
        return caller(self.RESOURCES[resource], reason, **params)


class Validated:
    def __init__(self, checks: List[tuple], subject: str):
        self.checks, self.subject = checks, subject
        self.failing = [name for name, ok, _ in checks if not ok]
        self.valid = not self.failing

    def __repr__(self):
        return f"<Validated {self.subject}: {'VALID' if self.valid else self.failing}>"


def validate_kpi_result(computed: Dict[str, Any],
                        compared: Optional[Dict[str, Any]] = None) -> Validated:
    ev = computed.get("evidence", [])
    checks = [
        ("value is a finite number",
         isinstance(computed.get("value"), (int, float)) and math.isfinite(computed["value"]),
         f"value={computed.get('value')}"),
        ("the formula used is recorded", bool(computed.get("formula")),
         computed.get("formula", "")),
        ("the figure cites its source system", bool(computed.get("source_system")),
         computed.get("source_system", "")),
        ("raw inputs are recorded", bool(computed.get("inputs_used")),
         str(computed.get("inputs_used", ""))),
        ("every department in the evidence is identified",
         bool(ev) and all(e.get("dept_code") for e in ev), f"{len(ev)} evidence rows"),
        ("evidence covers exactly the departments computed over",
         len(ev) == computed.get("n_departments"), f"{len(ev)} rows"),
        ("provisional figures carry a caveat",
         computed.get("figure_status") == "final" or bool(computed.get("caveat")),
         f"figure_status={computed.get('figure_status')}"),
    ]
    if compared is not None:
        checks.append(("the target names the authority that set it",
                       bool(compared.get("target_source")), compared.get("target_source", "")))
    return Validated(checks, computed.get("kpi_id", "?"))


# =============================================================================


Overwriting edss_core.py


### Cell 2 · The protocol

A connector is a **protocol**, not a helper function. Modelled on MCP:

* the server **advertises** its resources (`describe()` — capability discovery),
* the client calls them **by name**, holding a scoped token,
* every response uses the same `{ok, resource, served_by, data, meta}` envelope,
* every failure uses the same error vocabulary — `UNAUTHORIZED`, `FORBIDDEN_SCOPE`, `UNKNOWN_RESOURCE`, `BAD_REQUEST`, `NOT_FOUND`, `BACKEND_FAILURE`,
* and every call — allowed or refused — lands in one log.

In [2]:
# =============================================================================
# Cell 2  ·  The protocol — one typed door for every source
# =============================================================================
# Concept 5: a connector is a PROTOCOL, not a helper function. Everything that
# reaches a source goes through the same envelope, the same auth, the same
# error codes and the same log — whether the source behind it is a file, a
# database or an HTTP API.
#
# Modelled on MCP: the server advertises its resources (capability discovery),
# the client calls them by name, and the transport shape never changes.

import csv, io, json, re, sqlite3, textwrap, time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

from edss_core import (DOMAINS, KPI_DEFINITIONS, REPORTING_PERIOD,
                       DEPARTMENT_REPORTS_CSV, NUMERIC_COLUMNS, BENCHMARK_API_DB,
                       compute_kpi, compare_to_target, validate_kpi_result)

# ---- error codes: one vocabulary for every backend ------------------------
E_UNKNOWN_RESOURCE = "UNKNOWN_RESOURCE"
E_UNAUTHORIZED     = "UNAUTHORIZED"
E_FORBIDDEN_SCOPE  = "FORBIDDEN_SCOPE"
E_BAD_REQUEST      = "BAD_REQUEST"
E_NOT_FOUND        = "NOT_FOUND"
E_BACKEND_FAILURE  = "BACKEND_FAILURE"


class MCPError(Exception):
    def __init__(self, code: str, message: str):
        super().__init__(f"{code}: {message}")
        self.code, self.message = code, message


@dataclass
class Resource:
    """One advertised capability of the server."""
    name: str
    description: str
    params: Dict[str, Any]          # typed parameter schema
    returns: str
    scope: str                      # the scope a token must hold
    backend: str                    # which backend currently serves it
    handler: Callable = None

    def manifest(self) -> Dict[str, Any]:
        return {"name": self.name, "description": self.description,
                "params": self.params, "returns": self.returns,
                "required_scope": self.scope, "served_by": self.backend}


class MCPServer:
    """One server, many backends. Agents never see which is which."""

    def __init__(self, name: str):
        self.name = name
        self.resources: Dict[str, Resource] = {}
        self.tokens: Dict[str, Dict[str, Any]] = {}
        self.log: List[Dict[str, Any]] = []

    # ---- registration & auth -------------------------------------------
    def expose(self, resource: Resource) -> None:
        self.resources[resource.name] = resource

    def issue_token(self, token: str, holder: str, scopes: List[str]) -> None:
        self.tokens[token] = {"holder": holder, "scopes": set(scopes)}

    def describe(self) -> Dict[str, Any]:
        """Capability discovery — what a client may call, without calling it."""
        return {"server": self.name,
                "resources": [r.manifest() for r in self.resources.values()]}

    # ---- the single entry point ----------------------------------------
    def call(self, token: str, resource: str, **params) -> Dict[str, Any]:
        started = time.perf_counter()
        entry = {"seq": len(self.log) + 1, "resource": resource,
                 "holder": self.tokens.get(token, {}).get("holder", "<unknown>"),
                 "params": {k: (f"<{len(v)} rows>" if isinstance(v, list) else v)
                            for k, v in params.items()},
                 "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}

        def finish(ok, code=None, note=""):
            entry.update({"ok": ok, "error_code": code, "note": note,
                          "ms": round((time.perf_counter() - started) * 1000, 2)})
            self.log.append(entry)

        if token not in self.tokens:
            finish(False, E_UNAUTHORIZED)
            raise MCPError(E_UNAUTHORIZED, "token not recognised by this server")
        if resource not in self.resources:
            finish(False, E_UNKNOWN_RESOURCE)
            raise MCPError(E_UNKNOWN_RESOURCE,
                           f"'{resource}' is not exposed; call describe() to see "
                           f"{sorted(self.resources)}")
        res = self.resources[resource]
        if res.scope not in self.tokens[token]["scopes"]:
            finish(False, E_FORBIDDEN_SCOPE)
            raise MCPError(E_FORBIDDEN_SCOPE,
                           f"'{resource}' needs scope '{res.scope}'; this token holds "
                           f"{sorted(self.tokens[token]['scopes'])}")
        for p, spec in res.params.items():
            if spec.get("required") and p not in params:
                finish(False, E_BAD_REQUEST)
                raise MCPError(E_BAD_REQUEST, f"{resource}: missing parameter '{p}'")
        try:
            data = res.handler(**params)
        except MCPError as exc:
            finish(False, exc.code, exc.message)
            raise
        except Exception as exc:
            finish(False, E_BACKEND_FAILURE, str(exc))
            raise MCPError(E_BACKEND_FAILURE, f"{res.backend} failed: {exc}")
        entry["served_by"] = res.backend
        finish(True)
        # The envelope is identical whatever the backend was.
        return {"ok": True, "resource": resource, "served_by": res.backend, "data": data,
                "meta": {"server": self.name, "at": entry["at"]}}


class MCPClient:
    """What an agent holds. It knows a token and resource names — nothing else."""

    def __init__(self, server: MCPServer, token: str):
        self._server, self._token = server, token

    def describe(self) -> Dict[str, Any]:
        return self._server.describe()

    def read(self, resource: str, **params) -> Any:
        return self._server.call(self._token, resource, **params)["data"]


print("Protocol defined: typed resources, scoped tokens, one envelope, one error")
print("vocabulary, one log — regardless of whether a file, a database or an API")
print("sits behind the resource.")


Protocol defined: typed resources, scoped tokens, one envelope, one error
vocabulary, one log — regardless of whether a file, a database or an API
sits behind the resource.


### Cell 3 · Three real backends

A CSV on disk, a `sqlite3` database built and populated here, and a mock HTTP API that sleeps for its latency and fails like a web service. Different storage engines, different query languages, different failure modes — and Cell 5's agent will see none of that.

In [3]:
# =============================================================================
# Cell 3  ·  Three real backends — a file, a SQLite database, a mock API
# =============================================================================
# The lab reference is "one server exposing File + SQLite + API". These are not
# three wrappers around the same dict: a CSV is written to disk, a real sqlite3
# database is created and queried with SQL, and the API backend imposes latency
# and returns HTTP-style failures.

WORKDIR = Path("lab5_sources")
WORKDIR.mkdir(exist_ok=True)

# ---- backend 1: a FILE ----------------------------------------------------
CSV_PATH = WORKDIR / "department_reports_2025_26.csv"
CSV_PATH.write_text(DEPARTMENT_REPORTS_CSV)


def _cast(row: Dict[str, str]) -> Dict[str, Any]:
    out = dict(row)
    for col, caster in NUMERIC_COLUMNS.items():
        if col in out:
            out[col] = caster(out[col])
    return out


class FileBackend:
    name = "file:csv"

    def __init__(self, path: Path):
        self.path = path

    def department_reports(self, period: str, category: str = None) -> Dict[str, Any]:
        if period != REPORTING_PERIOD:
            raise MCPError(E_NOT_FOUND, f"no export on disk for '{period}'")
        with self.path.open() as fh:
            rows = [_cast(r) for r in csv.DictReader(fh)]
        if category:
            rows = [r for r in rows if r["category"] == category]
            if not rows:
                raise MCPError(E_NOT_FOUND, f"no departments in category '{category}'")
        return {"rows": rows, "row_count": len(rows), "period": period,
                "source_system": "Department reports export (CSV on disk)"}


# ---- backend 2: SQLite ----------------------------------------------------
DB_PATH = WORKDIR / "institution.db"
if DB_PATH.exists():
    DB_PATH.unlink()
conn = sqlite3.connect(DB_PATH)
conn.executescript("""
CREATE TABLE kpi_definition (
    kpi_id TEXT PRIMARY KEY, name TEXT, domain TEXT, unit TEXT, formula TEXT,
    target REAL, target_source TEXT, source_system TEXT, figure_status TEXT, caveat TEXT);
CREATE TABLE department_report (
    dept_code TEXT PRIMARY KEY, dept_name TEXT, category TEXT, period TEXT,
    payload TEXT);
""")
conn.executemany(
    "INSERT INTO kpi_definition VALUES (?,?,?,?,?,?,?,?,?,?)",
    [(k, d["name"], d["domain"], d["unit"], d["formula"], d["target"],
      d["target_source"], d["source_system"], d.get("figure_status", "final"),
      d.get("caveat")) for k, d in KPI_DEFINITIONS.items()])
conn.executemany(
    "INSERT INTO department_report VALUES (?,?,?,?,?)",
    [(r["dept_code"], r["dept_name"], r["category"], REPORTING_PERIOD, json.dumps(r))
     for r in (_cast(x) for x in csv.DictReader(io.StringIO(DEPARTMENT_REPORTS_CSV)))])
conn.commit()


class SQLiteBackend:
    name = "sqlite"

    def __init__(self, path: Path):
        self.path = path

    def _rows(self, sql: str, args=()) -> List[sqlite3.Row]:
        c = sqlite3.connect(self.path)
        c.row_factory = sqlite3.Row
        try:
            return c.execute(sql, args).fetchall()
        finally:
            c.close()

    def kpi_definitions(self, kpi_ids: List[str] = None) -> Dict[str, Any]:
        if kpi_ids:
            marks = ",".join("?" * len(kpi_ids))
            rows = self._rows(
                f"SELECT * FROM kpi_definition WHERE kpi_id IN ({marks})", kpi_ids)
            if len(rows) != len(kpi_ids):
                found = {r["kpi_id"] for r in rows}
                raise MCPError(E_NOT_FOUND,
                               f"unknown KPI id(s): {sorted(set(kpi_ids) - found)}")
        else:
            rows = self._rows("SELECT * FROM kpi_definition ORDER BY kpi_id")
        defs = {}
        for r in rows:
            d = dict(KPI_DEFINITIONS[r["kpi_id"]])   # formula fields for the calculator
            d.update({"name": r["name"], "domain": r["domain"], "target": r["target"],
                      "target_source": r["target_source"],
                      "source_system": r["source_system"]})
            defs[r["kpi_id"]] = d
        return {"definitions": defs, "count": len(defs),
                "source_system": "Institutional KPI register (SQLite)"}

    # The SAME resource the file backend serves — used in Cell 6 to swap sources.
    def department_reports(self, period: str, category: str = None) -> Dict[str, Any]:
        sql = "SELECT payload FROM department_report WHERE period = ?"
        args = [period]
        if category:
            sql += " AND category = ?"
            args.append(category)
        rows = [json.loads(r["payload"]) for r in self._rows(sql, args)]
        if not rows:
            raise MCPError(E_NOT_FOUND,
                           f"no rows for period '{period}'"
                           + (f" and category '{category}'" if category else ""))
        return {"rows": rows, "row_count": len(rows), "period": period,
                "source_system": "Department reports (SQLite table)"}


# ---- backend 3: a mock HTTP API -------------------------------------------
class APIBackend:
    name = "http:benchmark-api"
    LATENCY_S = 0.02

    def external_benchmark(self, kpi_id: str) -> Dict[str, Any]:
        time.sleep(self.LATENCY_S)              # a network call is not free
        if kpi_id not in KPI_DEFINITIONS:
            raise MCPError(E_BAD_REQUEST, f"'{kpi_id}' is not a known KPI id")
        record = BENCHMARK_API_DB.get(kpi_id)
        if record is None:
            raise MCPError(E_NOT_FOUND,
                           f"no published external benchmark for {kpi_id} "
                           f"({KPI_DEFINITIONS[kpi_id]['name']})")
        return {"kpi_id": kpi_id, **record,
                "source_system": "External benchmark API (NIRF cohort)"}


file_backend, sqlite_backend, api_backend = (FileBackend(CSV_PATH),
                                             SQLiteBackend(DB_PATH), APIBackend())

print(f"  file   : {CSV_PATH}  ({CSV_PATH.stat().st_size:,} bytes on disk)")
print(f"  sqlite : {DB_PATH}  ({DB_PATH.stat().st_size:,} bytes, "
      f"{len(sqlite_backend._rows('SELECT kpi_id FROM kpi_definition'))} KPI rows, "
      f"{len(sqlite_backend._rows('SELECT dept_code FROM department_report'))} dept rows)")
print(f"  api    : mock HTTP benchmark service, {int(APIBackend.LATENCY_S * 1000)} ms "
      f"latency, {len(BENCHMARK_API_DB)} of {len(KPI_DEFINITIONS)} KPIs covered")


  file   : lab5_sources/department_reports_2025_26.csv  (1,000 bytes on disk)
  sqlite : lab5_sources/institution.db  (20,480 bytes, 9 KPI rows, 8 dept rows)
  api    : mock HTTP benchmark service, 20 ms latency, 3 of 9 KPIs covered


### Cell 4 · Expose, authorise, advertise

Chapter 15's `connector.expose` block, implemented literally. Note the scoped tokens: the **Academic Agent cannot reach external benchmarks**, and the intern token can read only the KPI register. Authorisation is data, not a comment.

In [4]:
# =============================================================================
# Cell 4  ·  Expose the sources, issue scoped tokens, advertise the capabilities
# =============================================================================
# Chapter 15's Lab 5 pseudocode, implemented literally:
#     connector.expose: read('institutional KPIs & dashboards')
#                       read('department reports')
#                       read('external benchmarks')
#     agents.use(connector)

server = MCPServer("edss-connector")

server.expose(Resource(
    name="department_reports",
    description="Per-department metric rows for one reporting period.",
    params={"period": {"type": "string", "required": True},
            "category": {"type": "string", "required": False,
                         "enum": ["circuit", "core"]}},
    returns="{rows, row_count, period, source_system}",
    scope="read:departments", backend=file_backend.name,
    handler=file_backend.department_reports))

server.expose(Resource(
    name="kpi_definitions",
    description="Approved institution KPI definitions: formula, unit, target, authority.",
    params={"kpi_ids": {"type": "array", "required": False}},
    returns="{definitions, count, source_system}",
    scope="read:kpi-register", backend=sqlite_backend.name,
    handler=sqlite_backend.kpi_definitions))

server.expose(Resource(
    name="external_benchmark",
    description="Published peer median for a KPI. Coverage is partial.",
    params={"kpi_id": {"type": "string", "required": True}},
    returns="{kpi_id, peer_median, cohort, year, source_system}",
    scope="read:benchmarks", backend=api_backend.name,
    handler=api_backend.external_benchmark))

# ---- scoped tokens: the Registrar's authorisation, expressed as data -------
server.issue_token("tok-coordinator", "apniLeap AURA Coordinator",
                   ["read:departments", "read:kpi-register", "read:benchmarks"])
server.issue_token("tok-academic-agent", "Academic Agent",
                   ["read:departments", "read:kpi-register"])   # no benchmark access
server.issue_token("tok-intern", "Summer intern (read-only KPI register)",
                   ["read:kpi-register"])

print("=" * 88)
print("CAPABILITY MANIFEST — what a client can discover without calling anything")
print("=" * 88)
manifest = server.describe()
print(f"  server: {manifest['server']}\n")
for r in manifest["resources"]:
    req = [p for p, s in r["params"].items() if s.get("required")]
    print(f"  {r['name']}")
    print(f"      {r['description']}")
    print(f"      params  : {list(r['params'])}   required: {req}")
    print(f"      returns : {r['returns']}")
    print(f"      scope   : {r['required_scope']}     served by: {r['served_by']}")

print("\n  tokens issued:")
for tok, rec in server.tokens.items():
    print(f"      {tok:<22} {rec['holder']:<42} {sorted(rec['scopes'])}")


CAPABILITY MANIFEST — what a client can discover without calling anything
  server: edss-connector

  department_reports
      Per-department metric rows for one reporting period.
      params  : ['period', 'category']   required: ['period']
      returns : {rows, row_count, period, source_system}
      scope   : read:departments     served by: file:csv
  kpi_definitions
      Approved institution KPI definitions: formula, unit, target, authority.
      params  : ['kpi_ids']   required: []
      returns : {definitions, count, source_system}
      scope   : read:kpi-register     served by: sqlite
  external_benchmark
      Published peer median for a KPI. Coverage is partial.
      params  : ['kpi_id']   required: ['kpi_id']
      returns : {kpi_id, peer_median, cohort, year, source_system}
      scope   : read:benchmarks     served by: http:benchmark-api

  tokens issued:
      tok-coordinator        apniLeap AURA Coordinator                  ['read:benchmarks', 'read:departments', 're

### Cell 5 · Agents use the client — and only the client

The Lab 2 KPI pipeline, re-run with every read going through MCP.

Read `kpi_pipeline` and notice what is **absent**: no file path, no SQL, no URL, no backend name. It knows resource names and nothing else — which is exactly why the next cell can move a source out from under it.

In [5]:
# =============================================================================
# Cell 5  ·  Agents use the client — and only the client
# =============================================================================
# The KPI pipeline from Lab 2, re-run with every read going through MCP. The
# agent code below contains no path, no SQL, no URL and no backend name: it
# knows resource names and nothing else.

client = MCPClient(server, "tok-coordinator")


def kpi_pipeline(client: MCPClient, period: str, kpi_ids: List[str]) -> List[Dict[str, Any]]:
    """An agent. Notice what is absent: no file path, no SQL, no URL."""
    reports = client.read("department_reports", period=period)
    rows = reports["rows"]
    # Where the ROWS actually came from — distinct from the KPI register's
    # system-of-record, and the thing that changes when a source is swapped.
    rows_source = reports["source_system"]
    defs = client.read("kpi_definitions", kpi_ids=kpi_ids)["definitions"]
    out = []
    for kpi_id in kpi_ids:
        computed = compute_kpi(kpi_id, rows)
        compared = compare_to_target(kpi_id, computed["value"])
        gate = validate_kpi_result(computed, compared)
        assert gate.valid, f"validation gate failed for {kpi_id}: {gate.failing}"
        try:
            bench = client.read("external_benchmark", kpi_id=kpi_id)
        except MCPError as exc:
            bench = {"unavailable": exc.message}
        out.append({"kpi_id": kpi_id, "name": defs[kpi_id]["name"],
                    "value": computed["value"], "unit": computed["unit"],
                    "target": compared["target"], "status": compared["status"],
                    "source_system": computed["source_system"],
                    "rows_source": rows_source,
                    "benchmark": bench})
    return out


results = kpi_pipeline(client, REPORTING_PERIOD, ["KPI-P1", "KPI-R2", "KPI-P3"])

print("=" * 88)
print("THE PIPELINE, RUN ENTIRELY THROUGH THE CONNECTOR")
print("=" * 88)
for r in results:
    peer = (f"peer median {r['benchmark']['peer_median']}"
            if "peer_median" in r["benchmark"]
            else f"no benchmark — {r['benchmark']['unavailable'][:52]}…")
    print(f"  {r['kpi_id']:<8} {r['name']:<26} {r['value']:>8.2f} {r['unit']:<12} "
          f"{r['status']:<13} {peer}")

print(f"\n  connector calls made: {len(server.log)}")
print("  backends touched    : "
      f"{sorted({e.get('served_by') for e in server.log if e.get('served_by')})}")
print("  ...and the agent function above names none of them.")


THE PIPELINE, RUN ENTIRELY THROUGH THE CONNECTOR
  KPI-P1   Placement percentage          75.12 %            below target  peer median 78.6
  KPI-R2   Citations per faculty          7.26 per faculty  below target  peer median 11.4
  KPI-P3   Internship completion         73.38 %            below target  no benchmark — no published external benchmark for KPI-P3 (Internsh…

  connector calls made: 5
  backends touched    : ['file:csv', 'http:benchmark-api', 'sqlite']
  ...and the agent function above names none of them.


### Cell 6 · The acceptance test — swap a source, change no agent

`department_reports` moves from the CSV file to the SQLite table. Two lines change, **inside the connector**. `kpi_pipeline` is re-run as-is.

Every figure is identical to 1e-9. What *does* change is the provenance string — because the rows really did come from a different place, and the evidence trail has to say so.

In [6]:
# =============================================================================
# Cell 6  ·  The acceptance test — swap a source, change no agent
# =============================================================================
# "All data access goes through the connector; swapping a source needs no agent
# change."
#
# department_reports is currently served by the CSV file. Below it is re-pointed
# at the SQLite table — a different storage engine, a different query language,
# a different source_system string — and the SAME kpi_pipeline function is run
# again, untouched.

before_backend = server.resources["department_reports"].backend
before = kpi_pipeline(client, REPORTING_PERIOD, ["KPI-P1", "KPI-R2", "KPI-P3"])

# ---- the entire swap: two lines, inside the connector, agent-side unchanged
server.resources["department_reports"].handler = sqlite_backend.department_reports
server.resources["department_reports"].backend = sqlite_backend.name

after = kpi_pipeline(client, REPORTING_PERIOD, ["KPI-P1", "KPI-R2", "KPI-P3"])

print("=" * 88)
print("SOURCE SWAP — file:csv  ->  sqlite")
print("=" * 88)
print(f"  resource        : department_reports")
print(f"  backend before  : {before_backend}")
print(f"  backend after   : {server.resources['department_reports'].backend}")
print(f"  agent code edited: 0 lines  (kpi_pipeline was re-run as-is)\n")

print(f"  {'KPI':<8}{'before (CSV)':>16}{'after (SQLite)':>18}   identical")
same = True
for b, a in zip(before, after):
    ok = abs(b["value"] - a["value"]) < 1e-9
    same = same and ok
    print(f"  {b['kpi_id']:<8}{b['value']:>16.4f}{a['value']:>18.4f}   {ok}")

print(f"\n  every figure identical across the swap: {same}")
print(f"\n  what DID change — where the rows were actually read from:")
print(f"      before: {before[0]['rows_source']}")
print(f"      after : {after[0]['rows_source']}")
print(f"\n  what did NOT change — the KPI register's system-of-record, which is a")
print(f"  property of the metric, not of the storage engine:")
print(f"      {before[0]['source_system']}  (both runs)")
print("\n  That is the point: the agent is insulated from storage, but the evidence")
print("  trail still records truthfully where each number was read from.")


SOURCE SWAP — file:csv  ->  sqlite
  resource        : department_reports
  backend before  : file:csv
  backend after   : sqlite
  agent code edited: 0 lines  (kpi_pipeline was re-run as-is)

  KPI         before (CSV)    after (SQLite)   identical
  KPI-P1           75.1205           75.1205   True
  KPI-R2            7.2564            7.2564   True
  KPI-P3           73.3848           73.3848   True

  every figure identical across the swap: True

  what DID change — where the rows were actually read from:
      before: Department reports export (CSV on disk)
      after : Department reports (SQLite table)

  what did NOT change — the KPI register's system-of-record, which is a
  property of the metric, not of the storage engine:
      Placement Cell tracker  (both runs)

  That is the point: the agent is insulated from storage, but the evidence
  trail still records truthfully where each number was read from.


### Cell 7 · Governance, and the bypass

Eight access attempts: scope refusals, an unknown token, an unexposed resource, a missing parameter, and two genuine backend `NOT_FOUND`s. All arrive in the server's own error vocabulary rather than as stack traces.

Then the pitfall, demonstrated rather than described: an agent reads the CSV directly. The number it computes is **correct** — and completely ungoverned. No scope was checked, no provenance recorded, nothing logged.

That is the whole argument for the connector. The rule *"every figure in a published brief must correspond to a logged connector call"* is only enforceable because the log exists.

In [7]:
# =============================================================================
# Cell 7  ·  Governance — scopes, unknown resources, backend failures, bypass
# =============================================================================
# The pitfall: "No agent should touch a source directly; a bypass defeats
# governance later." A connector that can be walked around is decoration.

print("=" * 88)
print("ACCESS CONTROL")
print("=" * 88)

attempts = [
    ("intern reads the KPI register (allowed)", "tok-intern", "kpi_definitions", {}),
    ("intern reads department rows (no scope)", "tok-intern", "department_reports",
     {"period": REPORTING_PERIOD}),
    ("Academic Agent reads a benchmark (no scope)", "tok-academic-agent",
     "external_benchmark", {"kpi_id": "KPI-P1"}),
    ("unknown token", "tok-forged", "kpi_definitions", {}),
    ("unknown resource", "tok-coordinator", "student_medical_records", {}),
    ("missing required parameter", "tok-coordinator", "department_reports", {}),
    ("backend says not found", "tok-coordinator", "department_reports",
     {"period": "academic year 2019-20"}),
    ("benchmark genuinely absent", "tok-coordinator", "external_benchmark",
     {"kpi_id": "KPI-F1"}),
]
refusals = []
for label, tok, res, params in attempts:
    try:
        server.call(tok, res, **params)
        print(f"  [allowed ] {label}")
    except MCPError as exc:
        refusals.append((label, exc.code))
        print(f"  [{exc.code:<16}] {label}\n                     {exc.message}")

# ---- bypass detection ------------------------------------------------------
print("\n" + "=" * 88)
print("BYPASS DETECTION — an agent reaching around the connector")
print("=" * 88)
calls_before = len(server.log)
sneaky_rows = [_cast(r) for r in csv.DictReader(CSV_PATH.open())]   # direct file read
sneaky_value = compute_kpi("KPI-P1", sneaky_rows)["value"]
calls_after = len(server.log)

print(f"  an agent read {CSV_PATH.name} directly and computed KPI-P1 = {sneaky_value}")
print(f"  connector calls logged by that read: {calls_after - calls_before}")
print(f"  -> the figure is numerically right and completely ungoverned: no scope was")
print(f"     checked, no provenance recorded, nothing in the audit log.")
print(f"\n  Governance rule, enforceable because the log exists: every figure in a")
print(f"  published brief must correspond to a logged connector call.")

governed_ids = {e["resource"] for e in server.log if e["ok"]}
print(f"  resources with a logged, authorised read: {sorted(governed_ids)}")

print("\n" + "=" * 88)
print(f"CONNECTOR AUDIT LOG — {len(server.log)} calls")
print("=" * 88)
print(f"  {'#':<4}{'holder':<28}{'resource':<24}{'backend':<20}{'ok':<7}{'code'}")
for e in server.log:
    print(f"  {e['seq']:<4}{e['holder'][:26]:<28}{e['resource'][:22]:<24}"
          f"{str(e.get('served_by', '—'))[:18]:<20}{str(e['ok']):<7}{e['error_code'] or ''}")


ACCESS CONTROL
  [allowed ] intern reads the KPI register (allowed)
  [FORBIDDEN_SCOPE ] intern reads department rows (no scope)
                     'department_reports' needs scope 'read:departments'; this token holds ['read:kpi-register']
  [FORBIDDEN_SCOPE ] Academic Agent reads a benchmark (no scope)
                     'external_benchmark' needs scope 'read:benchmarks'; this token holds ['read:departments', 'read:kpi-register']
  [UNAUTHORIZED    ] unknown token
                     token not recognised by this server
  [UNKNOWN_RESOURCE] unknown resource
                     'student_medical_records' is not exposed; call describe() to see ['department_reports', 'external_benchmark', 'kpi_definitions']
  [BAD_REQUEST     ] missing required parameter
                     department_reports: missing parameter 'period'
  [NOT_FOUND       ] backend says not found
                     no rows for period 'academic year 2019-20'


  [NOT_FOUND       ] benchmark genuinely absent
                     no published external benchmark for KPI-F1 (Budget utilisation)

BYPASS DETECTION — an agent reaching around the connector
  an agent read department_reports_2025_26.csv directly and computed KPI-P1 = 75.1205
  connector calls logged by that read: 0
  -> the figure is numerically right and completely ungoverned: no scope was
     checked, no provenance recorded, nothing in the audit log.

  Governance rule, enforceable because the log exists: every figure in a
  published brief must correspond to a logged connector call.
  resources with a logged, authorised read: ['department_reports', 'external_benchmark', 'kpi_definitions']

CONNECTOR AUDIT LOG — 23 calls
  #   holder                      resource                backend             ok     code
  1   apniLeap AURA Coordinator   department_reports      file:csv            True   
  2   apniLeap AURA Coordinator   kpi_definitions         sqlite              True   
  

### Cell 8 · Acceptance checks — the rubric, executed

In [8]:
# =============================================================================
# Cell 8  ·  Acceptance checks — the rubric, executed
# =============================================================================
# Worklet 15, Lab 5 acceptance: "All data access goes through the connector;
# swapping a source needs no agent change."
# Pitfall guarded: "No agent should touch a source directly."

checks = []


def check(name, condition, evidence=""):
    checks.append((name, bool(condition), evidence))


served = {e.get("served_by") for e in server.log if e.get("served_by")}
check("E1  one envelope and one error vocabulary, whatever the backend",
      len(served) >= 3
      and all({"resource", "served_by", "data", "meta"} <= set(
          server.call("tok-coordinator", "kpi_definitions", kpi_ids=["KPI-P1"]))
          for _ in [0]),
      f"backends served across this session: {sorted(served)}; every response uses "
      f"the same {{ok, resource, served_by, data, meta}} envelope")

check("E2  a file, a real SQLite database and an API all served the pipeline",
      CSV_PATH.exists() and DB_PATH.exists()
      and {"file:csv", "sqlite", "http:benchmark-api"}
      <= {e.get("served_by") for e in server.log},
      f"{CSV_PATH.name} on disk, {DB_PATH.name} "
      f"({DB_PATH.stat().st_size:,} bytes), benchmark API called")

check("E3  capability discovery works without calling a resource",
      len(server.describe()["resources"]) == len(server.resources)
      and all("required_scope" in r for r in server.describe()["resources"]),
      f"describe() advertises {len(server.describe()['resources'])} resources "
      f"with params, returns and scope")

check("E4  swapping the source changed every figure by nothing",
      same and before_backend != server.resources["department_reports"].backend,
      f"{before_backend} -> {server.resources['department_reports'].backend}; "
      f"{len(before)} KPIs identical to 1e-9")

check("E5  the agent function was not edited for the swap",
      all(b["kpi_id"] == a["kpi_id"] for b, a in zip(before, after)),
      "kpi_pipeline was re-run as-is; the change was two lines inside the connector")

check("E6  provenance follows the actual read, not the agent's assumption",
      before[0]["rows_source"] != after[0]["rows_source"],
      f"'{before[0]['rows_source']}' -> '{after[0]['rows_source']}'")

check("E7  a token without the scope is refused",
      any(code == E_FORBIDDEN_SCOPE for _, code in refusals),
      f"{sum(1 for _, c in refusals if c == E_FORBIDDEN_SCOPE)} scope refusals "
      f"(intern -> departments, Academic Agent -> benchmarks)")

check("E8  an unknown token is refused",
      any(code == E_UNAUTHORIZED for _, code in refusals),
      "tok-forged rejected before any backend was reached")

check("E9  an unexposed resource cannot be reached at all",
      any(code == E_UNKNOWN_RESOURCE for _, code in refusals),
      "student_medical_records is not exposed, so it is not reachable")

check("E10 backend failures arrive in the server's own error vocabulary",
      any(code == E_NOT_FOUND for _, code in refusals),
      "a missing period and a missing benchmark both surfaced as NOT_FOUND, "
      "not as a stack trace")

check("E11 a missing required parameter is caught before the backend runs",
      any(code == E_BAD_REQUEST for _, code in refusals),
      "department_reports without 'period' refused at the door")

check("E12 every authorised call is logged with holder, resource and backend",
      server.log and all({"holder", "resource", "ok", "ms"} <= set(e) for e in server.log),
      f"{len(server.log)} calls logged, "
      f"{sum(1 for e in server.log if e['ok'])} authorised")

check("E13 a direct-to-source read produces no log entry — which is how a bypass "
      "is detected",
      calls_after == calls_before,
      f"the direct CSV read added {calls_after - calls_before} connector calls, so it "
      f"cannot be reconciled against the audit log")

print("=" * 88)
print("LAB 5 ACCEPTANCE CHECKS")
print("=" * 88)
for name, ok, ev in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}]  {name}")
    if ev:
        print(f"          {ev}")
failed = [n for n, ok, _ in checks if not ok]
print("=" * 88)
print(f"  {len(checks) - len(failed)}/{len(checks)} checks passed.")
assert not failed, f"FAILED: {failed}"
print("  Lab 5 acceptance criteria satisfied.")


LAB 5 ACCEPTANCE CHECKS
  [PASS]  E1  one envelope and one error vocabulary, whatever the backend
          backends served across this session: ['file:csv', 'http:benchmark-api', 'sqlite']; every response uses the same {ok, resource, served_by, data, meta} envelope
  [PASS]  E2  a file, a real SQLite database and an API all served the pipeline
          department_reports_2025_26.csv on disk, institution.db (20,480 bytes), benchmark API called
  [PASS]  E3  capability discovery works without calling a resource
          describe() advertises 3 resources with params, returns and scope
  [PASS]  E4  swapping the source changed every figure by nothing
          file:csv -> sqlite; 3 KPIs identical to 1e-9
  [PASS]  E5  the agent function was not edited for the swap
          kpi_pipeline was re-run as-is; the change was two lines inside the connector
  [PASS]  E6  provenance follows the actual read, not the agent's assumption
          'Department reports export (CSV on disk)' -> 'Depart

### Cell 9 · Deliverables

In [9]:
# =============================================================================
# Cell 9  ·  Deliverables
# =============================================================================
outdir = Path("lab5_artifacts")
outdir.mkdir(exist_ok=True)

artifacts = {
    "lab5_capability_manifest.json": server.describe(),
    "lab5_tokens_and_scopes.json": {t: {"holder": r["holder"],
                                        "scopes": sorted(r["scopes"])}
                                    for t, r in server.tokens.items()},
    "lab5_connector_audit_log.json": server.log,
    "lab5_swap_evidence.json": {
        "resource": "department_reports",
        "backend_before": before_backend,
        "backend_after": server.resources["department_reports"].backend,
        "agent_lines_changed": 0,
        "results_before": [{k: v for k, v in r.items() if k != "benchmark"}
                           for r in before],
        "results_after": [{k: v for k, v in r.items() if k != "benchmark"}
                          for r in after],
        "all_figures_identical": same},
    "lab5_refusals.json": [{"attempt": a, "code": c} for a, c in refusals],
    "lab5_acceptance_checks.json": [{"check": n, "passed": ok, "evidence": e}
                                    for n, ok, e in checks],
}
for fname, obj in artifacts.items():
    (outdir / fname).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str))

print("=" * 88)
print("ARTIFACTS WRITTEN")
print("=" * 88)
for fname in artifacts:
    print(f"  {outdir / fname}  ({(outdir / fname).stat().st_size:,} bytes)")
print(f"\n  Live sources also on disk:")
print(f"      {CSV_PATH}  ({CSV_PATH.stat().st_size:,} bytes)")
print(f"      {DB_PATH}  ({DB_PATH.stat().st_size:,} bytes)")
print(f"\n  Resources exposed   : {len(server.resources)}")
print(f"  Connector calls     : {len(server.log)} "
      f"({sum(1 for e in server.log if e['ok'])} authorised, "
      f"{sum(1 for e in server.log if not e['ok'])} refused)")
print(f"  Acceptance checks   : {sum(ok for _, ok, _ in checks)}/{len(checks)} passed")


ARTIFACTS WRITTEN
  lab5_artifacts/lab5_capability_manifest.json  (1,365 bytes)
  lab5_artifacts/lab5_tokens_and_scopes.json  (436 bytes)
  lab5_artifacts/lab5_connector_audit_log.json  (7,127 bytes)
  lab5_artifacts/lab5_swap_evidence.json  (1,923 bytes)
  lab5_artifacts/lab5_refusals.json  (575 bytes)
  lab5_artifacts/lab5_acceptance_checks.json  (2,576 bytes)

  Live sources also on disk:
      lab5_sources/department_reports_2025_26.csv  (1,000 bytes)
      lab5_sources/institution.db  (20,480 bytes)

  Resources exposed   : 3
  Connector calls     : 24 (14 authorised, 10 refused)
  Acceptance checks   : 13/13 passed


---

## Where Lab 5 leaves the worklet

**Nodes activated:** the connector sits under every data-facing agent.

**What exists now:** one governed door. Sources can be migrated, mocked or replaced without touching an agent; a token's scope decides what it can reach; and every read is attributable to a holder, a resource and a backend.

**What Lab 6 adds:** the runtime around the loop. The connector governs *reads*; Lab 6 governs *the run* — binding skills (Lab 3), memory (Lab 4) and the connector (Lab 5) behind one controller that enforces budgets, writes an audit log, and checkpoints so a crashed run resumes instead of restarting.

> Lab 6 acceptance (Chapter 15): *"Every action is logged with inputs/outputs; a crashed run resumes from its last checkpoint."*